## Random Forest

O [Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) é um modelo de ensemble que combina múltiplas árvores de decisão para melhorar a performance e reduzir o overfitting.

Cada árvore é treinada em uma amostra diferente dos dados, e a decisão final é feita por votação.

Esse modelo tende a ser mais robusto e apresentar melhor generalização em comparação com uma única árvore.

### Hiperparâmetros utilizados

- **n_estimators**: número de árvores na floresta.
  - Mais árvores → melhor desempenho (até certo ponto)

- **max_depth**: profundidade máxima das árvores.
  - Controla o overfitting

- **min_samples_split**: mínimo de amostras para dividir um nó

- **min_samples_leaf**: O número mínimo de amostras necessário para que um nó seja uma folha

### Estratégia

Foi utilizado GridSearchCV para encontrar a melhor combinação de hiperparâmetros, utilizando validação cruzada e a métrica ROC AUC.

In [1]:
import pandas as pd
import sys
import os

# add raiz do projeto
sys.path.append(os.path.abspath(".."))

from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from preprocessing.main_preprocessing import load_and_preprocess
from utils.experiment_logger import save_results


## BASELINE

In [2]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

smote_options = [False, True]

results = []


for scenario in scenarios:
    for use_smote in smote_options:

        print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

        X_train, X_test, y_train, y_test = load_and_preprocess(
            "../predictive_models/scrdata_202505.csv",
            scenario=scenario,
            use_smote=use_smote
        )

        steps = []
        
        if use_smote:
            steps.append(('smote', SMOTE(random_state=42)))
        steps.append(('rf', RandomForestClassifier(random_state=42, class_weight="balanced")))

        model = Pipeline(steps)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        results.append({
            "model": "RandomForest",
            "scenario": scenario,
            "smote": use_smote,
            "roc_auc": roc_auc_score(y_test, y_proba),
            "f1": f1_score(y_test, y_pred),
            "accuracy": accuracy_score(y_test, y_pred),
            "n_features": X_train.shape[1],
            "phase": "baseline",
            "timestamp": pd.Timestamp.now()
        })

#save_results(results, file_path="../results/experiments.csv")

df_result = pd.DataFrame(results)

display(df_result)



🔎 Scenario: sem_submodalidade | SMOTE: False

🔎 Scenario: sem_submodalidade | SMOTE: True

🔎 Scenario: submodalidade_agrupada | SMOTE: False

🔎 Scenario: submodalidade_agrupada | SMOTE: True

🔎 Scenario: submodalidade_engineered | SMOTE: False

🔎 Scenario: submodalidade_engineered | SMOTE: True


,model,scenario,smote,roc_auc,f1,accuracy,n_features,phase,timestamp
0,RandomForest,sem_submodalidade,False,0.930210,0.873364,0.852510,67,baseline,2026-04-19 12:34:29.811596
1,RandomForest,sem_submodalidade,True,0.929446,0.867909,0.849078,67,baseline,2026-04-19 12:35:05.879978
2,RandomForest,submodalidade_agrupada,False,0.943146,0.885355,0.866180,97,baseline,2026-04-19 12:35:33.081206
3,RandomForest,submodalidade_agrupada,True,0.942463,0.882779,0.865158,97,baseline,2026-04-19 12:36:09.963363
4,RandomForest,submodalidade_engineered,False,0.930210,0.873364,0.852510,67,baseline,2026-04-19 12:36:36.543812
5,RandomForest,submodalidade_engineered,True,0.929446,0.867909,0.849078,67,baseline,2026-04-19 12:37:12.490708


## GRIDSEARCH

In [3]:
X_train, X_test, y_train, y_test = load_and_preprocess(
    "../predictive_models/scrdata_202505.csv",
    scenario="sem_submodalidade",
    use_smote=False
)


param_grid_rf = {
    "smote": [SMOTE(random_state=42), "passthrough"],
    "rf__n_estimators": [100, 200],
    "rf__min_samples_split": [2, 5],
    "rf__min_samples_leaf": [1, 2]
}

grid_rf = GridSearchCV(
    estimator=Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42,
        n_jobs=-1))
    ]),
    param_grid=param_grid_rf,
    cv=5,
    scoring="roc_auc",
    n_jobs=1
)

grid_rf.fit(X_train, y_train)

print("Best parameters:", grid_rf.best_params_)
print("Best ROC AUC (CV):", grid_rf.best_score_)


#? BEST MODEL TEST EVALUATION

best_rf = grid_rf.best_estimator_

y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]


#? TUNING (CV)

tuning_results = []

cv_results = pd.DataFrame(grid_rf.cv_results_)
cv_results['smote_used'] = cv_results['param_smote'].apply(lambda x: x != 'passthrough')

for smote_val in [True, False]:
    sub_results = cv_results[cv_results['smote_used'] == smote_val]
    if not sub_results.empty:
        best_row = sub_results.sort_values('mean_test_score', ascending=False).iloc[0]
        params = best_row['params']
        
        tuning_results.append({
            "model": "RandomForest",
            "scenario": "sem_submodalidade",
            "smote": smote_val,
            "phase": "tuning_cv",
            "roc_auc": best_row['mean_test_score'],
            "f1": None,
            "accuracy": None,
            "best_params": str(params),
            "timestamp": pd.Timestamp.now()
        })
        
        # Re-fit the best model for this group to get test metrics
        best_model_group = grid_rf.estimator.set_params(**params)
        best_model_group.fit(X_train, y_train)
        
        y_pred = best_model_group.predict(X_test)
        y_proba = best_model_group.predict_proba(X_test)[:, 1]
        
        tuning_results.append({
            "model": "RandomForest",
            "scenario": "sem_submodalidade",
            "smote": smote_val,
            "phase": "test",
            "roc_auc": roc_auc_score(y_test, y_proba),
            "f1": f1_score(y_test, y_pred),
            "accuracy": accuracy_score(y_test, y_pred),
            "best_params": str(params),
            "timestamp": pd.Timestamp.now()
        })

save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))

Best parameters: {'rf__min_samples_leaf': 2, 'rf__min_samples_split': 2, 'rf__n_estimators': 200, 'smote': 'passthrough'}
Best ROC AUC (CV): 0.9321227871940081


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,RandomForest,sem_submodalidade,True,tuning_cv,0.930527,NaN,NaN,"{'rf__min_samples_leaf': 2, 'rf__min_samples_s...",2026-04-19 12:45:22.870973
1,RandomForest,sem_submodalidade,True,test,0.930803,0.868414,0.850173,"{'rf__min_samples_leaf': 2, 'rf__min_samples_s...",2026-04-19 12:45:36.419460
2,RandomForest,sem_submodalidade,False,tuning_cv,0.932123,NaN,NaN,"{'rf__min_samples_leaf': 2, 'rf__min_samples_s...",2026-04-19 12:45:36.420373
3,RandomForest,sem_submodalidade,False,test,0.932190,0.874057,0.853732,"{'rf__min_samples_leaf': 2, 'rf__min_samples_s...",2026-04-19 12:45:44.315994
